# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShivanshRastogi315/FlyRank_Internship-repo/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**The Baseline Rule:** 
We will rank pages based on a simple "Visibility Heuristic". Pages with high search volume and a strong average position are statistically most likely to trigger and capture AI Overviews/Traffic. 
*   **Score Formula:** `search_volume / avg_position` (Missing positions are penalized as rank 100).
*   **Reason Codes:**
    *   `R01`: Dominant Head Term (Volume > 1000 & Position <= 10)
    *   `R02`: Striking Distance (Volume > 1000 & Position > 10)
    *   `R03`: Long-Tail Performer (Volume <= 1000 & Position <= 10)
    *   `R04`: Low Signal (Everything else)

In [1]:
import pandas as pd
import numpy as np
import os

# 1. Load the data (using the correct relative path!)
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Clean inputs for the baseline rule
df['search_volume'] = df['search_volume'].fillna(0)
df['avg_position'] = df['avg_position'].fillna(100)

# 2. Define the scoring and reason code functions
def calculate_baseline_score(row):
    # Prevent division by zero just in case
    pos = row['avg_position'] if row['avg_position'] > 0 else 1
    return row['search_volume'] / pos

def assign_reason_code(row):
    vol = row['search_volume']
    pos = row['avg_position']
    
    if vol > 1000 and pos <= 10:
        return 'R01'
    elif vol > 1000 and pos > 10:
        return 'R02'
    elif vol <= 1000 and pos <= 10:
        return 'R03'
    else:
        return 'R04'

print("Baseline rule logic defined successfully.")

Baseline rule logic defined successfully.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# 1. Apply the rules
df['baseline_action_score'] = df.apply(calculate_baseline_score, axis=1)
df['baseline_reason_code'] = df.apply(assign_reason_code, axis=1)

# 2. Rank the queue (Sort descending by score)
ranked_queue = df.sort_values(by='baseline_action_score', ascending=False).copy()

# 3. Create the outputs directory if it doesn't exist, then save
output_dir = "../outputs"
os.makedirs(output_dir, exist_ok=True)
output_path = f"{output_dir}/baseline_action_score.csv"

# Select only the required columns for the handoff
columns_to_export = ['content_id', 'baseline_action_score', 'baseline_reason_code']
ranked_queue[columns_to_export].to_csv(output_path, index=False)

print(f"Ranked queue saved to: {output_path}")
print(f"Total rows ranked: {len(ranked_queue)}")

Ranked queue saved to: ../outputs/baseline_action_score.csv
Total rows ranked: 30000


## 3. Top-20 review
**Top 20 Review:** 
By reviewing the top 20 scored items alongside the *actual* target (`ai_sessions_90d`), we can immediately see the baseline's accuracy (Precision@20). If a row has a high baseline score but zero AI sessions, the heuristic failed on that specific item.

In [3]:
# Extract the Top 20
top_20 = ranked_queue.head(20)

# Display the Top 20 alongside the actual target to evaluate performance
review_columns = ['content_id', 'baseline_action_score', 'baseline_reason_code', 'search_volume', 'avg_position', 'ai_sessions_90d']
print("--- TOP 20 BASELINE PICKS ---")
display(top_20[review_columns])

# Calculate Baseline Precision@20
true_positives = (top_20['ai_sessions_90d'] > 0).sum()
precision_at_20 = (true_positives / 20) * 100
print(f"\nBaseline Precision@20: {precision_at_20:.2f}%")

--- TOP 20 BASELINE PICKS ---


,content_id,baseline_action_score,baseline_reason_code,search_volume,avg_position,ai_sessions_90d
15616,content_201a4a56f4d6,22200.000000,R01,22200.0,1.0,0
2281,content_004d8a5ce838,18100.000000,R01,18100.0,0.0,0
2991,content_6f6a4e56098c,10343.750000,R01,33100.0,3.2,0
28371,content_6e098546a7a2,7789.473684,R01,14800.0,1.9,0
11883,content_137b16f29ca9,5400.000000,R01,5400.0,0.0,0
13502,content_f76ccf7a7834,5210.526316,R01,49500.0,9.5,0
2074,content_6ef3dcb7be11,4516.666667,R01,27100.0,6.0,0
16736,content_e12868d1f396,4172.413793,R01,12100.0,2.9,0
17859,content_5c5fab9d41e7,4111.111111,R01,22200.0,5.4,0
25137,content_c35d34131e91,4000.000000,R01,14800.0,3.7,0



Baseline Precision@20: 0.00%


## 4. Weak picks + leakage check

**Leakage Check:** 
Confirmed clean. The baseline score strictly relies on `search_volume` and `avg_position`, both of which represent historical/current state prior to the 90-day AI traffic accumulation. The target (`ai_sessions_90d`) was absolutely not used in the calculation.

**Weak Picks Analysis:**
The baseline assumes that high traditional search visibility equals high AI visibility. This is a flawed assumption. Many top-ranked pages are likely generic navigational queries (where AI Overviews rarely trigger) or simple factual pages. The heuristic completely ignores content depth, engagement, and topic intent—which is exactly why we need a Machine Learning model to take over from here.

In [4]:
# Identify "Weak Picks" in the top 100 (High Score, but ZERO AI Traffic)
top_100 = ranked_queue.head(100)
weak_picks = top_100[top_100['ai_sessions_90d'] == 0]

print("--- WEAK PICKS ANALYSIS ---")
print(f"Out of the Top 100 baseline recommendations, {len(weak_picks)} actually have ZERO AI traffic.")
print("This represents the 'False Positive' rate of our simple heuristic.")

# Display a few examples of where the baseline failed
if not weak_picks.empty:
    print("\nExamples of Baseline Failures:")
    display(weak_picks[['content_id', 'baseline_reason_code', 'search_volume', 'avg_position', 'ai_sessions_90d']].head(5))

--- WEAK PICKS ANALYSIS ---
Out of the Top 100 baseline recommendations, 97 actually have ZERO AI traffic.
This represents the 'False Positive' rate of our simple heuristic.

Examples of Baseline Failures:


,content_id,baseline_reason_code,search_volume,avg_position,ai_sessions_90d
15616,content_201a4a56f4d6,R01,22200.0,1.0,0
2281,content_004d8a5ce838,R01,18100.0,0.0,0
2991,content_6f6a4e56098c,R01,33100.0,3.2,0
28371,content_6e098546a7a2,R01,14800.0,1.9,0
11883,content_137b16f29ca9,R01,5400.0,0.0,0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.